# AlphaFold Ensemble Competition - Tournament Edition

This notebook screens a list of potential peptide binders using a 'Winner Stays On' (King of the Hill) tournament structure.
It evaluates pairs of binders against the target protein. The winner advances to face the next candidate on the list.

In [1]:
import os
import time
import py3Dmol
import numpy as np
from pathlib import Path
from af_competition import run_colabfold_async, analyze_binding, process_ensemble, optimize_threshold  # noqa: F401


## 1. Configuration
Define the target, binding site, and list of potential binders.

In [2]:
TARGET_SEQ = "SQIPASEQETLVRPKPLLLKLLKSVGAQKDTYTMKEVLFYLGQYIMTKRLYDAAQQHIVYCSNDLLGDLFGVPSFSVKEHRKIYTMIYRNLVVVNQQ"
BINDING_SITE_RESIDUES = [42, 84]  # 1-indexed on Target Chain

# List of potential binders to screen in the tournament
BINDER_CANDIDATES = [
    "ETFSDLWKLLPE", # Candidate 0
    "LTFEHYWAQLTS", # Candidate 1
    "LTWEHYWAQLTS", 
    "LTFEHYLAQLTS", 
    "LTFEHIWAQLTS", 
    "LTFEHAFAQLTS", 
    "LTFEDYTAQFTS"
]

NUM_SEEDS = 20
BASE_OUTPUT_DIRECTORY = "./colabfold_results"
MIN_PLDDT = 80.0


## 2. Tournament Execution
Runs the 'Winner Stays On' tournament.

In [5]:
champion_idx = 0
champion_seq = BINDER_CANDIDATES[0]

last_match_dir = ""
last_match_stats = {}
winning_state_last_match = ""
majority_wins = (NUM_SEEDS // 2) + 1

print(f"Starting Tournament with {len(BINDER_CANDIDATES)} candidates. Early stop threshold: {majority_wins} wins.\n")

for challenger_idx in range(1, len(BINDER_CANDIDATES)):
    challenger_seq = BINDER_CANDIDATES[challenger_idx]
    run_name = f"tournament_match_{champion_idx}_vs_{challenger_idx}"
    
    print(f"--- Match {challenger_idx}: Champion [{champion_idx}] vs Challenger [{challenger_idx}] ---")
    
    # --- 1. Execute ColabFold Asynchronously ---
    process, ACTUAL_OUTPUT_DIR = run_colabfold_async(TARGET_SEQ, champion_seq, challenger_seq, BASE_OUTPUT_DIRECTORY, run_name=run_name, num_seeds=NUM_SEEDS)
    last_match_dir = ACTUAL_OUTPUT_DIR
    
    # --- 2. Monitor and Analyze in Real-Time ---
    analyzed_pdbs = set()
    all_results = []
    early_stop_triggered = False
    
    while process.poll() is None:
        current_pdbs = set(Path(ACTUAL_OUTPUT_DIR).glob("*.pdb"))
        new_pdbs = current_pdbs - analyzed_pdbs
        
        newly_analyzed = 0
        for pdb in new_pdbs:
            try:
                res = analyze_binding(str(pdb), BINDING_SITE_RESIDUES, 0.0)
                if res["mean_plddt"] >= MIN_PLDDT:
                    all_results.append(res)
                analyzed_pdbs.add(pdb)
                newly_analyzed += 1
            except Exception:
                # File might be partially written; skip and retry next loop
                pass
                
        if newly_analyzed > 0 and all_results:
            opt = optimize_threshold(all_results, min_thresh=5.0, max_thresh=15.0, step=0.5)
            stats = opt['stats']
            champ_wins = stats.get('lig1_wins', 0)
            challenger_wins = stats.get('lig2_wins', 0)
            
            print(f"Models parsed: {len(analyzed_pdbs)}/{NUM_SEEDS} | Champion: {champ_wins} | Challenger: {challenger_wins}")
            
            if champ_wins >= majority_wins or challenger_wins >= majority_wins:
                print("  -> Insurmountable lead detected! Terminating AlphaFold early...")
                process.terminate()
                early_stop_triggered = True
                break
                
        time.sleep(5)  # Wait 5 seconds before checking again
        
    # Parse any final straggler PDBs after process ends
    current_pdbs = set(Path(ACTUAL_OUTPUT_DIR).glob("*.pdb"))
    new_pdbs = current_pdbs - analyzed_pdbs
    for pdb in new_pdbs:
        try:
            res = analyze_binding(str(pdb), BINDING_SITE_RESIDUES, 0.0)
            if res["mean_plddt"] >= MIN_PLDDT:
                all_results.append(res)
        except Exception:
            pass
            
    if not all_results:
        print(f"Match void: No models passed pLDDT threshold {MIN_PLDDT}.")
        print(f"Champion [{champion_idx}] retains title by default.\n")
        winning_state_last_match = "Ligand 1"
        continue
        
    # Final optimization across all parsed results
    opt = optimize_threshold(all_results, min_thresh=5.0, max_thresh=15.0, step=0.5)
    stats = opt['stats']
    last_match_stats = stats
    
    champ_wins = stats.get('lig1_wins', 0)
    challenger_wins = stats.get('lig2_wins', 0)
    
    print(f"\nFinal Match Results (Valid Models: {len(all_results)})")
    print(f"Champion score: {champ_wins} | Challenger score: {challenger_wins}")
    
    if challenger_wins > champ_wins:
        print(f"Challenger [{challenger_idx}] defeats Champion [{champion_idx}]!")
        champion_idx = challenger_idx
        champion_seq = challenger_seq
        winning_state_last_match = "Ligand 2"
    elif champ_wins > challenger_wins:
        print(f"Champion [{champion_idx}] defends the title!")
        winning_state_last_match = "Ligand 1"
    else:
        # TIE BREAKER
        print("Scores tied! Proceeding to pLDDT tie-breaker...")
        if champ_wins == 0 and challenger_wins == 0:
            print("Neither ligand achieved exclusive binding in any valid models. Champion retains title by default.")
            winning_state_last_match = "Neither"
        else:
            champ_plddt = np.mean([r['mean_plddt'] for r in stats['lig1_results']])
            challenger_plddt = np.mean([r['mean_plddt'] for r in stats['lig2_results']])
            print(f"Champion Avg pLDDT: {champ_plddt:.1f} | Challenger Avg pLDDT: {challenger_plddt:.1f}")
            
            if challenger_plddt > champ_plddt:
                print(f"Challenger [{challenger_idx}] wins by tie-breaker!")
                champion_idx = challenger_idx
                champion_seq = challenger_seq
                winning_state_last_match = "Ligand 2"
            else:
                print(f"Champion [{champion_idx}] wins by tie-breaker!")
                winning_state_last_match = "Ligand 1"
    print("\n")
    
print("=== TOURNAMENT COMPLETE ===")
print(f"ULTIMATE CHAMPION: Candidate [{champion_idx}] ({champion_seq})")


Starting Tournament with 7 candidates.

--- Match 1: Champion [0] vs Challenger [1] ---


E0000 00:00:1788533864.679779 2732885 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788533864.686600 2732885 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788533864.703930 2732885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788533864.703945 2732885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788533864.703948 2732885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788533864.703949 2732885 computation_placer.cc:177] computation placer already registered. Please check linka

2026-09-04 15:57:47,651 Running colabfold 1.6.1 (de5ab5f795ed95c70a7a9b6a9dc6bb5625016142)
2026-09-04 15:57:48,027 Running on GPU
2026-09-04 15:57:48,201 Found 5 citations for tools or databases
2026-09-04 15:57:48,201 Query 1/1: complex (length 121)


COMPLETE: 100%|██████████| 450/450 [elapsed: 00:01 remaining: 00:00]


2026-09-04 15:57:52,112 Setting max_seq=508, max_extra_seq=1015
2026-09-04 16:00:08,869 alphafold2_multimer_v3_model_1_seed_553887 recycle=0 pLDDT=80.7 pTM=0.768 ipTM=0.606
2026-09-04 16:01:39,642 alphafold2_multimer_v3_model_1_seed_553887 recycle=1 pLDDT=84 pTM=0.794 ipTM=0.718 tol=0.968
2026-09-04 16:01:40,378 alphafold2_multimer_v3_model_1_seed_553887 recycle=2 pLDDT=84.3 pTM=0.792 ipTM=0.73 tol=0.845
2026-09-04 16:01:41,113 alphafold2_multimer_v3_model_1_seed_553887 recycle=3 pLDDT=85.4 pTM=0.81 ipTM=0.753 tol=0.351
2026-09-04 16:01:41,113 alphafold2_multimer_v3_model_1_seed_553887 took 222.3s (3 recycles)
2026-09-04 16:01:41,861 alphafold2_multimer_v3_model_1_seed_553888 recycle=0 pLDDT=79.4 pTM=0.751 ipTM=0.527
2026-09-04 16:01:42,597 alphafold2_multimer_v3_model_1_seed_553888 recycle=1 pLDDT=85.3 pTM=0.8 ipTM=0.735 tol=2.7
2026-09-04 16:01:43,332 alphafold2_multimer_v3_model_1_seed_553888 recycle=2 pLDDT=85.2 pTM=0.803 ipTM=0.753 tol=0.413
2026-09-04 16:01:43,332 alphafold2_mult

...

2026-09-04 16:24:21,348 Running colabfold 1.6.1 (de5ab5f795ed95c70a7a9b6a9dc6bb5625016142)
2026-09-04 16:24:21,686 Running on GPU
2026-09-04 16:24:21,853 Found 5 citations for tools or databases
2026-09-04 16:24:21,853 Query 1/1: complex (length 121)


SUBMIT:   0%|          | 0/450 [elapsed: 00:00 remaining: ?]

2026-09-04 16:24:22,455 Sleeping for 9s. Reason: PENDING
2026-09-04 16:24:32,102 Sleeping for 10s. Reason: RUNNING


COMPLETE: 100%|██████████| 450/450 [elapsed: 00:11 remaining: 00:00]


2026-09-04 16:24:44,325 Sleeping for 10s. Reason: PENDING
2026-09-04 16:24:56,678 Setting max_seq=508, max_extra_seq=1015
2026-09-04 16:27:11,362 alphafold2_multimer_v3_model_1_seed_972837 recycle=0 pLDDT=80.9 pTM=0.766 ipTM=0.651
2026-09-04 16:28:42,629 alphafold2_multimer_v3_model_1_seed_972837 recycle=1 pLDDT=82.6 pTM=0.781 ipTM=0.704 tol=2.28
2026-09-04 16:28:43,382 alphafold2_multimer_v3_model_1_seed_972837 recycle=2 pLDDT=84.5 pTM=0.798 ipTM=0.74 tol=0.951
2026-09-04 16:28:44,126 alphafold2_multimer_v3_model_1_seed_972837 recycle=3 pLDDT=85 pTM=0.809 ipTM=0.752 tol=0.315
2026-09-04 16:28:44,127 alphafold2_multimer_v3_model_1_seed_972837 took 221.0s (3 recycles)
2026-09-04 16:28:44,886 alphafold2_multimer_v3_model_1_seed_972838 recycle=0 pLDDT=81.5 pTM=0.768 ipTM=0.654
2026-09-04 16:28:45,636 alphafold2_multimer_v3_model_1_seed_972838 recycle=1 pLDDT=84.1 pTM=0.805 ipTM=0.742 tol=1.35
2026-09-04 16:28:46,388 alphafold2_multimer_v3_model_1_seed_972838 recycle=2 pLDDT=85 pTM=0.805 i

## 3. Visualization of the Final Match
Renders the highest confidence model of the Ultimate Champion winning its last match.

In [6]:
if not last_match_stats:
    print("No valid models were generated to visualize.")
elif winning_state_last_match == "Neither":
    print("Neither ligand bound in the final match, nothing to visualize.")
else:
    winning_results = []
    
    if winning_state_last_match == "Ligand 1":
        winning_results = last_match_stats.get('lig1_results', [])
    elif winning_state_last_match == "Ligand 2":
        winning_results = last_match_stats.get('lig2_results', [])
        
    if not winning_results:
        print("No structural models available for the winning state to render.")
    else:
        best_model = max(winning_results, key=lambda x: x["mean_plddt"])
        best_pdb = os.path.join(last_match_dir, best_model["pdb_file"])
        
        print(f"Visualizing Final Match. Winning State: {winning_state_last_match}")
        print(f"Best Model: {best_model['pdb_file']} (pLDDT: {best_model['mean_plddt']:.1f})")
        print("Target (Chain A) = Grey | Ligand 1 (Champion) = Blue | Ligand 2 (Challenger) = Red")
        
        if os.path.exists(best_pdb):
            with open(best_pdb, 'r') as f:
                pdb_data = f.read()
                
            view = py3Dmol.view(width=800, height=600)
            view.addModel(pdb_data, 'pdb')
            
            # Target (Chain A)
            view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgray'}})
            # Ligand 1 (Chain B)
            view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'blue'}, 'stick': {'color': 'blue'}})
            # Ligand 2 (Chain C)
            view.setStyle({'chain': 'C'}, {'cartoon': {'color': 'red'}, 'stick': {'color': 'red'}})
            
            view.zoomTo()
            view.show()
        else:
            print(f"Error: Cannot find {best_pdb} to visualize.")


Visualizing Final Match. Winning State: Ligand 1
Best Model: complex_unrelaxed_rank_005_alphafold2_multimer_v3_model_1_seed_972842.pdb (pLDDT: 86.4)
Target (Chain A) = Grey | Ligand 1 (Champion) = Blue | Ligand 2 (Challenger) = Red


3Dmol.js failed to load for some reason. Please check your browser console for error messages.